In [1]:
import numpy as np
from scipy.stats import unitary_group
from scipy.linalg import eigvalsh
import matplotlib.pyplot as plt

In [2]:
I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0,-1j],[1j, 0]], dtype=complex)
Z  = np.array([[1, 0], [0,-1]], dtype=complex)
PAULIS = [I2, X, Y, Z]          # index: 0=I, 1=X, 2=Y, 3=Z

In [3]:
# ── Build n-qubit Pauli tensor product ─────────────────────────
def build_noise_operator(error_labels):
    """
    error_labels: list of ints in {0,1,2,3}, one per qubit
    returns: tensor product σ_{e[0]} ⊗ σ_{e[1]} ⊗ ... as (d x d) matrix
    """
    op = PAULIS[error_labels[0]]
    for e in error_labels[1:]:
        op = np.kron(op, PAULIS[e])
    return op

# ── Sample one error per qubit (non-Markovian step) ────────────
def sample_errors(last_errors, q, p_vec):
    """
    last_errors: np.array of ints, shape (n_qubits,)
    q: memory parameter in [0, 1]
    p_vec: np.array of shape (4,), Pauli probabilities [p0,px,py,pz]
    returns: new_errors, shape (n_qubits,)
    """
    n = len(last_errors)
    new_errors = np.empty(n, dtype=int)
    for k in range(n):
        if np.random.rand() < q:
            new_errors[k] = last_errors[k]          # copy last error
        else:
            new_errors[k] = np.random.choice(4, p=p_vec)  # fresh draw
    return new_errors

def trace_dist_to_maximally_mixed(rho):
    """
    Computes ||rho - I/d||_1 = sum_k |lambda_k - 1/d|
    Uses eigvalsh (faster, exploits hermiticity).
    """
    d = rho.shape[0]
    eigvals = eigvalsh(rho)          # real eigenvalues, sorted ascending
    return 0.5 * np.sum(np.abs(eigvals - 1.0 / d))

def prob_list(px, py, pz):
    return np.array([1-(px+py+pz), px, py, pz])

In [4]:
def build_single_qubit_full_channel(rho_q, conditional_probs):
    out = np.zeros((2, 2), dtype=complex)
    for mu, p in enumerate(conditional_probs):
        P = PAULIS[mu]
        out += p * (P @ rho_q @ P.conj().T)
    return out

def conditional_probs(last_error, q, p_vec):
    probs = (1 - q) * p_vec.copy()
    probs[last_error] += q
    return probs

def precompute_pauli_strings(n_qubits):
    """
    Precompute all 4^n_qubits tensor-product Pauli operators, shape (4^n, d, d).
    Call once per n_qubits value and reuse across all steps/trajectories.
    """
    d = 2 ** n_qubits
    n_strings = 4 ** n_qubits
    ops = np.empty((n_strings, d, d), dtype=complex)
    for i, indices in enumerate(np.ndindex(*(4,) * n_qubits)):
        N = PAULIS[indices[0]]
        for k in range(1, n_qubits):
            N = np.kron(N, PAULIS[indices[k]])
        ops[i] = N
    return ops  # shape (4^n, d, d)

def apply_local_channel_vectorized(rho, qubit_channels, pauli_ops):
    """
    Vectorized n-qubit Pauli channel using precomputed pauli_ops (4^n, d, d).
    qubit_channels: list of n arrays of shape (4,)
    """
    # Joint probability for each Pauli string via successive outer products
    probs = qubit_channels[0]
    for ch in qubit_channels[1:]:
        probs = np.outer(probs, ch).ravel()   # shape (4^n,)

    # Batched:  NRN[i] = pauli_ops[i] @ rho @ pauli_ops[i]†
    NR  = pauli_ops @ rho                                        # (4^n, d, d)
    NRN = NR @ pauli_ops.conj().transpose(0, 2, 1)              # (4^n, d, d)

    # Weighted sum over all Pauli strings
    return np.einsum('i,ijk->jk', probs, NRN)

def run_trajectory(rho0, n_qubits, t_steps, q, p_vec, pauli_ops):
    d = 2 ** n_qubits
    rho = rho0.astype(complex).copy()

    last_errors = np.array([np.random.choice(4, p=p_vec) for _ in range(n_qubits)])

    for _ in range(t_steps):
        U = unitary_group.rvs(d)
        rho = U @ rho @ U.conj().T

        qubit_channels = []
        new_last_errors = np.empty(n_qubits, dtype=int)
        for k in range(n_qubits):
            cprobs = conditional_probs(last_errors[k], q, p_vec)
            qubit_channels.append(cprobs)
            new_last_errors[k] = np.random.choice(4, p=cprobs)
        last_errors = new_last_errors

        rho = apply_local_channel_vectorized(rho, qubit_channels, pauli_ops)

    eigvals = eigvalsh(rho)
    return 0.5 * np.sum(np.abs(eigvals - 1.0 / d))

def estimate_trace_distance(rho0, n_qubits, t_steps, q, p_vec, n_traj, pauli_ops):
    samples = np.array([
        run_trajectory(rho0, n_qubits, t_steps, q, p_vec, pauli_ops)
        for _ in range(n_traj)
    ])
    return np.mean(samples), np.std(samples, ddof=1) / np.sqrt(n_traj)


In [5]:
def run_trajectory_v2(rho0, n_qubits, t_steps, q, p_vec, pauli_ops, unitaries):
    """
    Single trajectory for approach 2:
    - Uses precomputed Haar unitaries (shared across trajectories)
    - Samples one specific Pauli error per qubit per step
    - Applies that single Pauli operator (no channel averaging)
    Returns the final pure-path density matrix.
    """
    rho = rho0.astype(complex).copy()
    last_errors = np.array([np.random.choice(4, p=p_vec) for _ in range(n_qubits)])

    for t in range(t_steps):
        # Shared Haar unitary
        U = unitaries[t]
        rho = U @ rho @ U.conj().T

        # Sample one error per qubit from the non-Markovian conditional
        new_errors = np.empty(n_qubits, dtype=int)
        for k in range(n_qubits):
            cprobs = conditional_probs(last_errors[k], q, p_vec)
            new_errors[k] = np.random.choice(4, p=cprobs)
        last_errors = new_errors

        # Apply the single sampled Pauli string
        N = build_noise_operator(new_errors)
        rho = N @ rho @ N.conj().T

    return rho


def estimate_trace_distance_v2(rho0, n_qubits, t_steps, q, p_vec, pauli_ops, n_traj, unitaries):
    """
    Averages the final density matrices over n_traj trajectories (all sharing
    the same Haar unitaries), then computes ||rho_avg - I/d||_1.
    """
    d = 2 ** n_qubits
    rho_avg = np.zeros((d, d), dtype=complex)
    for _ in range(n_traj):
        rho_avg += run_trajectory_v2(rho0, n_qubits, t_steps, q, p_vec, pauli_ops, unitaries)
    rho_avg /= n_traj
    eigvals = eigvalsh(rho_avg)
    return 0.5 * np.sum(np.abs(eigvals - 1.0 / d))


In [6]:
q_values_v2  = [0.0, 0.3, 0.6, 0.9, 1.0]
t_max_v2     = 15
t_range_v2   = np.arange(1, t_max_v2 + 1)
n_traj_v2    = 400   # trajectories per unitary set
n_unitary_sets = 5  # independent Haar unitary realizations to average over

d_sys = 2 ** num_qubit

fig, ax = plt.subplots(figsize=(8, 5))

for q in q_values_v2:
    # dists_all[u, t] = trace distance for unitary set u at time step t
    dists_all = np.zeros((n_unitary_sets, len(t_range_v2)))

    for u in range(n_unitary_sets):
        # Fresh set of Haar unitaries for this realization
        unitaries = [unitary_group.rvs(d_sys) for _ in range(t_max_v2)]

        for i, t in enumerate(t_range_v2):
            dists_all[u, i] = estimate_trace_distance_v2(
                rho0, num_qubit, t, q, prob, pauli_ops, n_traj_v2, unitaries[:t]
            )

    means  = dists_all.mean(axis=0)
    stderr = dists_all.std(axis=0, ddof=1) / np.sqrt(n_unitary_sets)

    line, = ax.plot(t_range_v2, means, marker='o', markersize=3, label=f'q = {q}')
    ax.fill_between(t_range_v2, means - stderr, means + stderr, alpha=0.2, color=line.get_color())

ax.set_xlabel('Time step')
ax.set_ylabel(r'$\mathbb{E}_U\|\bar{\rho}_t - I/d\|_1$')
ax.set_title(f'trace distance averaged over {n_unitary_sets} unitary sets ({num_qubit} qubits)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'montecarlo_plots/num_qubits={num_qubit}_num_traj={n_traj_v2}_unitary={n_unitary_sets}.png')
plt.show()


NameError: name 'num_qubit' is not defined

In [ ]:
from scipy.optimize import curve_fit

def exp_decay(t, A, lam):
    return A * np.exp(-lam * t)

q_dense_v2    = np.linspace(0.05, 0.95, 20)
n_unitary_fit = 5
n_traj_fit    = 200

fit_lambdas_v2 = []
valid_qs_v2    = []

for q in q_dense_v2:
    # Average trace distances over multiple unitary sets
    means_q = np.zeros(len(t_range_v2))
    for _ in range(n_unitary_fit):
        unitaries = [unitary_group.rvs(d_sys) for _ in range(t_max_v2)]
        for i, t in enumerate(t_range_v2):
            means_q[i] += estimate_trace_distance_v2(
                rho0, num_qubit, t, q, prob, pauli_ops, n_traj_fit, unitaries[:t]
            )
    means_q /= n_unitary_fit

    try:
        popt, _ = curve_fit(exp_decay, t_range_v2, means_q, p0=[means_q[0], 0.3], maxfev=5000)
        fit_lambdas_v2.append(popt[1])
        valid_qs_v2.append(q)
    except RuntimeError:
        pass

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(valid_qs_v2, fit_lambdas_v2, 'o-', color='steelblue')
ax.set_xlabel('Memory parameter q')
ax.set_ylabel('Decay coefficient λ')
ax.set_title(f'Approach 2: exponential decay rate vs q ({num_qubit} qubits)')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
